# Latent-LoRA against the certified switch: five tasks, three seeds

**A comparison run, nothing to edit.** Five SuperNI tasks are learned one after another (t5-base, 10 epochs, 800 training
examples per task, seeds 42, 43, 44), once with **our method** (`ContinualModel` of `certified_switch.py`, rule "certified";
from the same adapters also "nearest" and "all" = plain adapters) and once with **Latent-LoRA** (Azghan et al., arXiv:2607.23837,
*re-implemented from the paper's text*, because no official code exists; rank 32, alpha 16, lambda 0.05, Gaussian-mixture router
with K=5). Both go through the same data loading, training loop (AdamW, constant learning rate 3e-4, effective batch 32), greedy
decoding and Rouge-L scoring (`run_five.py`). After every task all test inputs of all tasks learned so far are answered again, with the
original definition and with three reworded definitions, and every answer is saved.

**How to use it on Kaggle:** right-hand panel, *Session options*: Accelerator **GPU T4 x2** (one GPU also works), **Internet on**
(needed for the data repository and for the t5-base weights; if it is off, the notebook stops at once and says so). Then
**Save Version -> Save & Run All**. With two GPUs, two (method, seed) jobs run at the same time, one per GPU, each as its own
process with its own log file.

**What it writes** (all under `/kaggle/working/latent_lora_five/`): `results/<method>_seed<seed>.json` (every answer, after every task),
`logs/*.log`, `summary.json`, and at the end `/kaggle/working/latent_lora_five_small_files.zip` with all `.json` and `.log` files (no model weights).
Every `run_five.py` process first runs a self-test on the real device (two tiny tasks per method); if it fails, nothing else is started.

In [ ]:
import os, sys, re, json, time, shutil, subprocess, threading, zipfile

NB_TEST = os.environ.get("LL_NB_TEST") == "1"          # only used to test this notebook on a CPU; has no effect on Kaggle
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.environ.get("LL_NB_OUT") or "/kaggle/working/latent_lora_five"
ZIP = OUT.rstrip("/") + "_small_files.zip"
CODE = os.path.join(OUT, "code")
LOGS = os.path.join(OUT, "logs")
WORK = os.environ.get("LL_NB_WORK") or ("/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp") + "/latent_lora_work"
CFG = dict(
    model="t5-base",
    seeds=[42, 43, 44],
    methods=["ours", "latent_lora"],
    repo_url="https://github.com/liangyanshuo/gainlora",
    repo_commit="e9d0d700538c8d12892f7fd52b216291f7e42acc",     # the version of the data repository used for the earlier runs
    minutes_per_job=45,                                          # rough guess for one (method, seed) on a T4, see the estimate below
    progress_every_s=180,
)
if NB_TEST:
    CFG.update(seeds=[42], progress_every_s=20)
os.makedirs(CODE, exist_ok=True); os.makedirs(LOGS, exist_ok=True); os.makedirs(WORK, exist_ok=True)
print("results are written to:", OUT)

## The code
These cells hold the Python files; the next cell writes them to disk. `certified_switch.py` and `rewordings.json` are copies of the
project's files (not edited); `latent_lora.py` and `run_five.py` are the new files.

In [ ]:
FILES = {}

In [ ]:
FILES["certified_switch.py"] = r'''"""The certified switch as stand-alone code, for any model that has an embedding table and linear layers.

Nothing here depends on one architecture or on the code of another paper. Three parts:

  Zones          decides, from the frozen word-vectors of an input, which task the input is given to.
                 Each task has a protected zone that is never changed; the oldest zone that contains
                 an input owns it for ever; an input outside every zone goes to the nearest zone.
  RoutedLoRA     one low-rank adapter per task on a linear layer. Older adapters are frozen. For every
                 input only the adapters of tasks 1..route are switched on.
  ContinualModel puts both on a Hugging Face model (encoder-decoder or decoder-only), learns tasks one
                 after another and answers inputs.

Why an answer inside a zone can never change: the route of such an input is the oldest zone that
contains it. Later tasks only add newer zones, so the route stays the same, and the adapters it
switches on are frozen. This holds for any model and any later tasks.

Two rules are offered (`rule=` of ContinualModel):
  "certified"    the rule above. A new task whose instruction lies inside an older zone is refused on those
                 inputs (they stay with the older task); `learn` reports how many.
  "exact_first"  an input that carries the instruction of a task token for token goes to that task (the
                 oldest, if several tasks have the same instruction); otherwise as above. A sibling of an old
                 task can then be learned. The guarantee is weaker: an answer inside a zone stays the same
                 unless a task is learned later whose instruction that input carries exactly. An input that
                 carries a learned instruction exactly keeps its answer without conditions.
On tasks whose instructions do not collide the two rules give the same routes.

What it needs: every task's inputs start with the same text (its instruction). That opening is
what the zone is built from.
"""
import math

import torch
import torch.nn as nn


# ----------------------------------------------------------------------------------------------
# 1. Which task does an input belong to?
# ----------------------------------------------------------------------------------------------
def common_prefix(rows):
    """Number of tokens that all rows share at their start."""
    n = 0
    while all(len(r) > n for r in rows) and len({r[n] for r in rows}) == 1:
        n += 1
    return n


class Zones:
    """Protected zones built from a frozen embedding table.

    table    the model's input embedding matrix (vocabulary x dimensions); it is only read
    kappa    radius of a zone = kappa x distance from its signature to the nearest signature known
             when the task is added (unrelated tasks and earlier tasks)
    a        a token counts with weight a / (a + its share among the tokens of unrelated text)
    """

    def __init__(self, table, pad_id, kappa=0.5, a=1e-3, head=64):
        self.table = table.detach().double()
        self.pad, self.kappa, self.a, self.head = pad_id, kappa, a, head
        self.weight = None
        self.tasks = []                                  # one dict per task, oldest first; never changed

    def _mean(self, rows, n=None):
        """Weighted average word-vector of the first n tokens of every row (all tokens if n is None)."""
        out = []
        for r in rows:
            ids = torch.as_tensor(r[:n] if n else r, dtype=torch.long, device=self.table.device)
            w = (ids != self.pad).double()
            if self.weight is not None:
                w = w * self.weight[ids]
            out.append((self.table[ids] * w[:, None]).sum(0) / w.sum().clamp(min=1e-12))
        return torch.stack(out)

    @staticmethod
    def _whitener(x):
        """Centre and rescale so that unrelated inputs vary by about the same amount in every direction."""
        mu = x.mean(0)
        cov = (x - mu).T @ (x - mu) / (x.shape[0] - 1)
        cov = 0.5 * cov + 0.5 * torch.eye(cov.shape[0], dtype=cov.dtype, device=cov.device) * torch.trace(cov) / cov.shape[0]
        w, v = torch.linalg.eigh(cov)
        return mu, v @ torch.diag(w ** -0.5) @ v.T

    def fit_background(self, unrelated_tasks):
        """unrelated_tasks: for each unrelated task, a list of token-id rows (its inputs, instruction included).
        Sets the token weights, the scale of distances and the signatures of the unrelated tasks. Done once."""
        rows = [r for task in unrelated_tasks for r in task]
        if self.a > 0:
            counts = torch.zeros(self.table.shape[0], dtype=torch.double, device=self.table.device)
            for r in rows:
                ids = torch.as_tensor([i for i in r if i != self.pad], dtype=torch.long, device=self.table.device)
                counts.index_add_(0, ids, torch.ones(len(ids), dtype=torch.double, device=self.table.device))
            self.weight = self.a / (self.a + counts / counts.sum())
            self.weight[self.pad] = 0.0
        self.mu_head, self.w_head = self._whitener(self._mean(rows, self.head))
        self.mu_all, self.w_all = self._whitener(self._mean(rows))
        self.known = torch.stack([self._signature(task[0][:common_prefix(task[:20])]) for task in unrelated_tasks])

    def _signature(self, opening):
        return (self._mean([opening])[0] - self.mu_head) @ self.w_head

    def add_task(self, train_rows, opening_len=None):
        """Make the zone of a new task from its training inputs. Returns how many of them were already inside
        an older zone (those inputs stay with the older task: the price of the guarantee, visible at once)."""
        n = opening_len or common_prefix(train_rows)
        assert n >= 3, "the task's inputs do not start with a common instruction"
        blocked = int(self.route(train_rows)[1].sum()) if self.tasks else 0
        sig = self._signature(train_rows[0][:n])
        known = torch.cat([self.known] + [t["sig"][None] for t in self.tasks])
        centre = ((self._mean(train_rows) - self.mu_all) @ self.w_all).mean(0)
        self.tasks.append({"n": n, "sig": sig, "rho": self.kappa * float((known - sig).norm(dim=1).min()), "centre": centre,
                           "opening": list(train_rows[0][:n])})
        return blocked

    def route(self, rows, rule="certified"):
        """For every row: its route (1-based task number; adapters 1..route are used) and whether it is certified.
        rule: 'certified' (oldest zone, else nearest zone), 'exact_first' (the oldest task whose instruction the
        input carries token for token, else as 'certified'), 'nearest' (nearest zone), 'centre' (nearest centre of
        the whole input), 'all' (the newest task: every adapter on)."""
        T = len(self.tasks)
        route = torch.full((len(rows),), T, dtype=torch.long)
        cert = torch.zeros(len(rows), dtype=torch.bool)
        if rule == "all" or T == 0:
            return route, cert
        if rule == "centre":
            whole = (self._mean(rows) - self.mu_all) @ self.w_all
            return torch.stack([(whole - t["centre"]).norm(dim=1) for t in self.tasks], 1).argmin(1).cpu() + 1, cert
        means = {n: self._mean(rows, n) for n in {t["n"] for t in self.tasks}}
        rel = torch.stack([(((means[t["n"]] - self.mu_head) @ self.w_head) - t["sig"]).norm(dim=1) / t["rho"]
                           for t in self.tasks], 1).cpu()        # 1 or less: inside that zone
        nearest = rel.argmin(1) + 1
        if rule in ("certified", "exact_first"):
            for k in range(T):                               # oldest first: an older claim is never overruled
                new = (rel[:, k] <= 1.0) & ~cert
                route[new] = k + 1
                cert |= new
        route = torch.where(cert, route, nearest)
        if rule == "exact_first":
            for i, row in enumerate(rows):                   # an exact claim comes before every zone; the oldest exact claim wins
                for k, t in enumerate(self.tasks):
                    if len(row) >= t["n"] and list(row[:t["n"]]) == t["opening"]:
                        route[i], cert[i] = k + 1, True
                        break
        return route, cert


# ----------------------------------------------------------------------------------------------
# 2. One adapter per task, switched on and off per input
# ----------------------------------------------------------------------------------------------
class Router:
    """Shared by all adapted layers of a model: the route of every input of the current batch."""
    route = None                                             # None: every adapter on


class RoutedLoRA(nn.Module):
    """A linear layer plus one low-rank adapter per task: y = W x + (alpha / r) * sum_k on_k(x) B_k A_k x."""

    def __init__(self, base, router, rank=8, alpha=16):
        super().__init__()
        self.base, self.router, self.rank, self.scale = base, router, rank, alpha / rank
        linear = isinstance(base, nn.Linear)                 # the other kind (GPT-2's Conv1D) stores the weight as in x out
        self.n_in = base.in_features if linear else base.weight.shape[0]
        self.n_out = base.out_features if linear else base.weight.shape[1]
        self.A, self.B = nn.ParameterList(), nn.ParameterList()

    def new_task(self):
        for p in list(self.A) + list(self.B):
            p.requires_grad_(False)                          # older adapters are never changed again
        ref = self.base.weight
        a = torch.empty(self.rank, self.n_in, device=ref.device, dtype=ref.dtype)
        nn.init.kaiming_uniform_(a, a=math.sqrt(5))
        self.A.append(nn.Parameter(a))
        self.B.append(nn.Parameter(torch.zeros(self.n_out, self.rank, device=ref.device, dtype=ref.dtype)))

    def forward(self, x):
        out = self.base(x)
        route = self.router.route
        for k, (a, b) in enumerate(zip(self.A, self.B)):
            delta = (x @ a.T) @ b.T * self.scale
            if route is not None:                            # adapter k+1 is on for inputs whose route is at least k+1
                on = (route >= k + 1).to(delta.dtype).view(-1, *([1] * (delta.dim() - 1)))
                assert on.shape[0] == delta.shape[0], "one route per input of the batch is needed"
                delta = delta * on
            out = out + delta
        return out


ATTENTION_NAMES = ("q", "v", "q_proj", "v_proj", "query", "value", "c_attn")


def add_adapters(model, router, rank=8, alpha=16, names=ATTENTION_NAMES):
    """Freeze the model and wrap its attention projections. Returns the wrapped layers."""
    for p in model.parameters():
        p.requires_grad_(False)
    wrapped = []
    for parent in list(model.modules()):
        for name, child in list(parent.named_children()):
            if name in names and hasattr(child, "weight") and child.weight.dim() == 2 and not isinstance(child, RoutedLoRA):
                layer = RoutedLoRA(child, router, rank, alpha)
                setattr(parent, name, layer)
                wrapped.append(layer)
    assert wrapped, "no attention projection found; pass the layer names of this model in `names`"
    return wrapped


# ----------------------------------------------------------------------------------------------
# 3. A model that learns tasks one after another
# ----------------------------------------------------------------------------------------------
class ContinualModel:
    def __init__(self, model, tokenizer, unrelated_tasks, kappa=0.5, a=1e-3, rank=8, alpha=16, max_len=256, max_new=32, rule="certified"):
        """unrelated_tasks: a few lists of input texts of tasks that will never be learned (they set the scale).
        rule: "certified" or "exact_first" (see the top of this file)."""
        assert rule in ("certified", "exact_first")
        self.model, self.tok, self.max_len, self.max_new, self.rule = model, tokenizer, max_len, max_new, rule
        self.seq2seq = bool(getattr(model.config, "is_encoder_decoder", False))
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.router = Router()
        self.layers = add_adapters(model, self.router, rank, alpha)
        self.zones = Zones(model.get_input_embeddings().weight, tokenizer.pad_token_id, kappa, a)
        self.zones.fit_background([self.ids(texts) for texts in unrelated_tasks])
        self.device = next(model.parameters()).device

    def ids(self, texts):
        return self.tok(list(texts), max_length=self.max_len, truncation=True)["input_ids"]

    def _pad(self, rows, left=False):
        width = max(len(r) for r in rows)
        ids = torch.full((len(rows), width), self.tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(rows), width), dtype=torch.long)
        for i, r in enumerate(rows):
            sl = slice(width - len(r), width) if left else slice(0, len(r))
            ids[i, sl] = torch.tensor(r)
            att[i, sl] = 1
        return ids.to(self.device), att.to(self.device)

    def learn(self, inputs, targets, epochs=3, batch=16, lr=1e-3, seed=0, log=None):
        """Learn one new task. Returns the number of training inputs that an older zone had already claimed."""
        rows = self.ids(inputs)
        blocked = self.zones.add_task(rows)
        for layer in self.layers:
            layer.new_task()
        route = self.zones.route(rows, self.rule)[0]         # inputs that stay with an older task do not train the new adapter
        answers = self.tok(list(targets), max_length=self.max_new, truncation=True, add_special_tokens=self.seq2seq)["input_ids"]
        if not self.seq2seq:
            answers = [a + [self.tok.eos_token_id] for a in answers]
        params = [p for p in self.model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr)
        gen = torch.Generator().manual_seed(seed)
        self.model.train()
        step = 0
        for _ in range(epochs):
            order = torch.randperm(len(rows), generator=gen).tolist()
            for s in range(0, len(order), batch):
                pick = order[s:s + batch]
                self.router.route = route[pick].to(self.device)
                if self.seq2seq:
                    ids, att = self._pad([rows[i] for i in pick])
                    lab, lab_att = self._pad([answers[i] for i in pick])
                    loss = self.model(input_ids=ids, attention_mask=att, labels=lab.masked_fill(lab_att == 0, -100)).loss
                else:
                    ids, att = self._pad([rows[i] + answers[i] for i in pick])
                    lab = ids.masked_fill(att == 0, -100)
                    for j, i in enumerate(pick):
                        lab[j, :len(rows[i])] = -100         # the loss is on the answer only
                    loss = self.model(input_ids=ids, attention_mask=att, labels=lab).loss
                loss.backward()
                opt.step()
                opt.zero_grad()
                step += 1
                if log and step % log == 0:
                    print(f"      step {step}: loss {loss.item():.3f}", flush=True)
        self.router.route = None
        return blocked

    @torch.no_grad()
    def answer(self, inputs, rule=None, route=None, batch=16):
        """Returns the answers, the route of every input and whether it is certified. rule: the model's own rule
        if not given; any rule of Zones.route can be asked for, to compare."""
        rows = self.ids(inputs)
        if route is None:
            route, cert = self.zones.route(rows, rule or self.rule)
        else:
            cert = torch.zeros(len(rows), dtype=torch.bool)
        self.model.eval()
        out = []
        for s in range(0, len(rows), batch):
            part = rows[s:s + batch]
            ids, att = self._pad(part, left=not self.seq2seq)
            self.router.route = route[s:s + batch].to(self.device)
            gen = self.model.generate(input_ids=ids, attention_mask=att, max_new_tokens=self.max_new, do_sample=False,
                                      num_beams=1, pad_token_id=self.tok.pad_token_id)
            if not self.seq2seq:
                gen = gen[:, ids.shape[1]:]
            out += [t.strip() for t in self.tok.batch_decode(gen, skip_special_tokens=True)]
        self.router.route = None
        return out, route, cert
'''

In [ ]:
FILES["latent_lora.py"] = r'''"""Latent-LoRA (Azghan et al., arXiv:2607.23837) as a re-implementation from the paper's text, in the style of
certified_switch.py: a class with `learn(inputs, targets, epochs, batch, lr, seed)` and `answer(inputs, rule=...)`.
There is no official code (checked, see README.md), so every detail the paper does not give is a choice, listed in README.md.

Parts:
  LatentLoRALayer   a frozen linear layer plus one trainable r x r matrix R_t per task in the frozen SVD subspace
                    of the layer's weight:  W + (alpha / r) * U_r Sigma_r R(x) V_r^T            (paper Eq. 5 and 15)
  GMMRouter         training-free task recognition: mean-pooled frozen embedding table, K-means mixture per task,
                    one covariance shared by all tasks, uniform prior                           (paper Eq. 11 to 14)
  LatentLoRAModel   learn / answer for an encoder-decoder Hugging Face model (T5). No stored examples, no trained router.
  CertifiedOnDevice the project's ContinualModel (certified_switch.py, not edited) with the model on a GPU, the zone
                    computations kept on the CPU in float64, and gradient accumulation. Same training loop as above.

Devices. Every route / zone / router computation is made on the CPU in float64 and only the final per-batch weights are
moved to the model's device. The model is built on the CPU and moved to the device once, by the constructors here.
"""
import gc
import os
import sys

import torch
import torch.nn as nn

try:
    from certified_switch import ATTENTION_NAMES, ContinualModel
except ImportError:                                           # running inside the repository: ../certified_switch
    sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "certified_switch"))
    from certified_switch import ATTENTION_NAMES, ContinualModel


# ----------------------------------------------------------------------------------------------
# 0. What both methods share: the training loop (same for both, so that the comparison is fair)
# ----------------------------------------------------------------------------------------------
class _Shared:
    """Needs self.model, self.tok, self.max_len, self.max_new, self.device, self.seq2seq, self.micro."""

    def _targets(self, targets):
        """Token ids of the answers, as in ContinualModel.learn (encoder-decoder branch)."""
        return self.tok(list(targets), max_length=self.max_new, truncation=True, add_special_tokens=True)["input_ids"]

    def _step(self, pick, rows, answers, set_route, micro, extra):
        """One optimiser step's gradients for the examples `pick`, in pieces of `micro` examples (gradient accumulation).
        The loss of a piece is the mean over its answer tokens (as in ContinualModel.learn), weighted by its share of the batch."""
        total, n = 0.0, len(pick)
        for c in range(0, n, micro):
            part = pick[c:c + micro]
            set_route(part)
            ids, att = self._pad([rows[i] for i in part])                 # created on self.device
            lab, lab_att = self._pad([answers[i] for i in part])
            loss = self.model(input_ids=ids, attention_mask=att, labels=lab.masked_fill(lab_att == 0, -100)).loss
            (loss * (len(part) / n)).backward()
            total += loss.item() * len(part) / n
        extra_value = None
        if extra is not None:                                             # a term that is added once per step (Latent-LoRA's Eq. 8)
            e = extra()
            e.backward()
            extra_value = e.item()
        return total, extra_value

    def _fit(self, rows, answers, set_route, epochs, batch, lr, seed, log=None, extra=None):
        """AdamW, constant learning rate, `batch` examples per optimiser step (split in pieces of self.micro examples; if the GPU
        runs out of memory the pieces are halved and the step is repeated). Returns the mean loss of the last epoch and, if
        `extra` is given, the mean of that term."""
        params = [p for p in self.model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr)
        gen = torch.Generator().manual_seed(seed)
        self.model.train()
        step, last, last_extra = 0, [], []
        for epoch in range(epochs):
            order = torch.randperm(len(rows), generator=gen).tolist()
            last, last_extra = [], []
            for s in range(0, len(order), batch):
                pick = order[s:s + batch]
                while True:
                    failed = False
                    try:
                        value, ev = self._step(pick, rows, answers, set_route, self.micro, extra)
                    except torch.cuda.OutOfMemoryError:
                        failed = True
                    if not failed:
                        break
                    opt.zero_grad(set_to_none=True)                       # outside the except block, so that the memory can be freed
                    gc.collect()
                    torch.cuda.empty_cache()
                    if self.micro <= 1:
                        raise RuntimeError("out of GPU memory even with one example at a time")
                    self.micro = max(1, self.micro // 2)
                    print(f"      GPU out of memory: now {self.micro} examples at a time (batch of {batch} unchanged)", flush=True)
                opt.step()
                opt.zero_grad()
                step += 1
                last.append(value)
                if ev is not None:
                    last_extra.append(ev)
                if log and step % log == 0:
                    print(f"      step {step}: loss {value:.3f}" + (f", orthogonal term {ev:.4g}" if ev is not None else ""), flush=True)
        self.model.eval()
        mean = lambda v: sum(v) / len(v) if v else None
        return mean(last), mean(last_extra)


def to_device_once(model, device):
    """The one place where the base model is moved to the device."""
    model.to(device)
    return torch.device(device) if not isinstance(device, torch.device) else device


# ----------------------------------------------------------------------------------------------
# 1. The project's method, on a device
# ----------------------------------------------------------------------------------------------
class CertifiedOnDevice(_Shared, ContinualModel):
    """ContinualModel (certified_switch.py, unchanged) used with the model on a GPU.

    Why a subclass: ContinualModel builds `Zones` from the model's embedding table at construction, so a model that is already
    on a GPU would make the zone computations run on the GPU in float64, and `self.device` is read at construction. Here the
    model is built on the CPU (zones, background and whitening are made on the CPU in float64), then moved once to the device,
    and `self.device` is set afterwards. Adapters made later by `new_task` are created on the device of the weight they wrap.
    `learn` has the same steps as ContinualModel.learn (zone, new adapters, route of the training inputs, AdamW, same
    batches) but goes through the shared loop with gradient accumulation; with micro >= batch the result is identical
    (checked in test_equivalence.py)."""

    def __init__(self, model, tokenizer, unrelated_tasks, device="cpu", micro=16, **kw):
        model = model.cpu()
        super().__init__(model, tokenizer, unrelated_tasks, **kw)
        assert self.seq2seq, "this wrapper is written for encoder-decoder models (T5)"
        self.device = to_device_once(model, device)
        self.micro = micro

    def learn(self, inputs, targets, epochs=3, batch=16, lr=1e-3, seed=0, log=None):
        rows = self.ids(inputs)
        blocked = self.zones.add_task(rows)                              # CPU, float64
        for layer in self.layers:
            layer.new_task()                                             # on the weight's device
        route = self.zones.route(rows, self.rule)[0]                     # CPU
        answers = self._targets(targets)

        def set_route(part):
            self.router.route = route[part].to(self.device)              # CPU -> device (long tensor, one per input)

        self.info = dict(zip(("loss", "extra"), self._fit(rows, answers, set_route, epochs, batch, lr, seed, log)))
        self.router.route = None
        return blocked


# ----------------------------------------------------------------------------------------------
# 2. Latent-LoRA
# ----------------------------------------------------------------------------------------------
class LatentRouter:
    """Shared by all adapted layers: the weight of every learned task's adapter for each input of the current batch."""
    p = None                                                              # (batch, tasks) on the model's device; None: no adapter


class LatentLoRALayer(nn.Module):
    """A frozen linear layer W (m x n) plus one trainable r x r matrix R_t per task (paper Sec. 3.2, LoRA-XS):
        W ~ U_r diag(S_r) V_r^T  (rank-r truncated SVD, computed once, frozen)
        y = W x + (alpha / r) * U_r diag(S_r) R(x) V_r^T x,      R(x) = sum_t p_t(x) R_t          (Eq. 5, 15)"""

    def __init__(self, base, router, rank=32, alpha=16):
        super().__init__()
        assert isinstance(base, nn.Linear)
        self.base, self.router, self.rank, self.scale = base, router, rank, alpha / rank
        W = base.weight.detach().cpu().double()                           # the SVD is made once, on the CPU, in float64
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        dt = base.weight.dtype
        self.register_buffer("U", U[:, :rank].to(dt).contiguous())        # m x r   frozen
        self.register_buffer("S", S[:rank].to(dt).contiguous())           # r       frozen
        self.register_buffer("V", Vh[:rank].T.to(dt).contiguous())        # n x r   frozen
        self.R = nn.ParameterList()

    def new_task(self):
        for p in self.R:
            p.requires_grad_(False)                                       # older adapters are snapshots, never changed again
        ref = self.base.weight
        self.R.append(nn.Parameter(torch.zeros(self.rank, self.rank, device=ref.device, dtype=ref.dtype)))

    def ortho(self):
        """Sum over older tasks i of || (S R_i)^T (S R_t) ||_F^2 for the newest task t (paper Eq. 8 and 10)."""
        if len(self.R) < 2:
            return self.S.sum() * 0.0
        new = self.S[:, None] * self.R[-1]
        return sum(((self.S[:, None] * old).T @ new).pow(2).sum() for old in list(self.R)[:-1])

    def forward(self, x):
        out = self.base(x)
        p = self.router.p
        if p is None or len(self.R) == 0:
            return out
        assert p.shape == (x.shape[0], len(self.R)), "one weight per learned task and per input of the batch is needed"
        Rs = torch.stack(list(self.R))                                    # (T, r, r)
        Rx = torch.einsum("bt,tij->bij", p, Rs)                           # R(x), one r x r matrix per input
        z = torch.einsum("blj,bij->bli", x @ self.V, Rx) * self.S         # Sigma R(x) V^T x
        return out + (z @ self.U.T) * self.scale


def add_latent_adapters(model, router, rank=32, alpha=16, names=ATTENTION_NAMES):
    """Freeze the model and wrap its query and value projections (all attention blocks of an encoder-decoder model)."""
    for p in model.parameters():
        p.requires_grad_(False)
    wrapped = []
    for parent in list(model.modules()):
        for name, child in list(parent.named_children()):
            if name in names and isinstance(child, nn.Linear) and not isinstance(child, LatentLoRALayer):
                layer = LatentLoRALayer(child, router, rank, alpha)
                setattr(parent, name, layer)
                wrapped.append(layer)
    assert wrapped, "no attention projection found"
    return wrapped


def _kmeans(x, k, iters, gen):
    """K-means with k-means++ seeding and `iters` Lloyd steps. Returns the centres and the share of points of every
    non-empty cluster."""
    n = x.shape[0]
    k = min(k, n)
    first = int(torch.randint(n, (1,), generator=gen))
    idx, d2 = [first], ((x - x[first]) ** 2).sum(1)
    for _ in range(1, k):
        prob = d2 / d2.sum() if float(d2.sum()) > 0 else torch.full_like(d2, 1.0 / n)
        j = int(torch.multinomial(prob, 1, generator=gen))
        idx.append(j)
        d2 = torch.minimum(d2, ((x - x[j]) ** 2).sum(1))
    mu = x[idx].clone()
    for _ in range(iters):
        assign = torch.cdist(x, mu).argmin(1)
        for c in range(k):
            m = assign == c
            if m.any():
                mu[c] = x[m].mean(0)
    counts = torch.bincount(torch.cdist(x, mu).argmin(1), minlength=k).double()
    keep = counts > 0
    return mu[keep], counts[keep] / n


class GMMRouter:
    """Training-free router (paper Sec. 3.3). Everything is on the CPU in float64.
    phi(x) = mean of the frozen input-embedding vectors of the tokens of x (Eq. 11); per task K Gaussians with K-means
    means and weights (Eq. 12), one covariance C for all tasks and components (Eq. 13), posterior with a uniform prior (Eq. 14)."""

    def __init__(self, table, pad_id, k=5, eps=0.01, iters=30):
        self.table = table.detach().cpu().double()                        # a copy: later changes of the model cannot reach it
        self.pad, self.k, self.eps, self.iters = pad_id, k, eps, iters
        self.scatter = None                                               # sum over tasks of S_t (Eq. 13 numerator)
        self.count = 0                                                    # sum over tasks of n_t
        self.tasks = []                                                   # per task: component means (K, d) and log weights (K,)
        self.chol = None

    def phi(self, rows):
        out = []
        for r in rows:
            ids = torch.as_tensor([i for i in r if i != self.pad], dtype=torch.long)
            out.append(self.table[ids].mean(0))
        return torch.stack(out)

    def add_task(self, rows, seed=0):
        """Fit the mixture of a new task on its training inputs and update the shared covariance. Old data are not needed."""
        x = self.phi(rows)
        mu, pi = _kmeans(x, self.k, self.iters, torch.Generator().manual_seed(seed))
        centred = x - x.mean(0)
        s = centred.T @ centred
        self.scatter = s if self.scatter is None else self.scatter + s
        self.count += x.shape[0]
        d = x.shape[1]
        C = self.scatter / self.count + self.eps * torch.eye(d, dtype=x.dtype)
        self.chol = torch.linalg.cholesky(C)
        self.tasks.append((mu, pi.log()))

    def posterior(self, rows):
        """(inputs, tasks) matrix of p(t | x)."""
        x = self.phi(rows)
        L = self.chol
        zx = torch.linalg.solve_triangular(L, x.T, upper=False).T         # whitened inputs, so that Mahalanobis = Euclidean
        logp = []
        for mu, logpi in self.tasks:
            zm = torch.linalg.solve_triangular(L, mu.T, upper=False).T
            d2 = torch.cdist(zx, zm) ** 2                                 # (inputs, K)
            logp.append(torch.logsumexp(logpi[None] - 0.5 * d2, 1))       # constants shared by all tasks cancel in the posterior
        return torch.softmax(torch.stack(logp, 1), 1)


class LatentLoRAModel(_Shared):
    ids = ContinualModel.ids                                              # same tokenisation and padding as the project's method
    _pad = ContinualModel._pad

    def __init__(self, model, tokenizer, rank=32, alpha=16, lam=0.05, k=5, eps=0.01, max_len=256, max_new=32,
                 device="cpu", micro=16, ortho_reduce="sum"):
        """rank, alpha, lam, k, eps: see README.md. ortho_reduce: how the Eq. 8 terms of the many adapted layers are combined."""
        assert ortho_reduce in ("sum", "mean")
        model = model.cpu()
        self.model, self.tok, self.max_len, self.max_new = model, tokenizer, max_len, max_new
        self.seq2seq = bool(getattr(model.config, "is_encoder_decoder", False))
        assert self.seq2seq, "this implementation is written for encoder-decoder models (T5)"
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.lam, self.ortho_reduce, self.micro = lam, ortho_reduce, micro
        self.router = LatentRouter()
        self.layers = add_latent_adapters(model, self.router, rank, alpha)
        self.gmm = GMMRouter(model.get_input_embeddings().weight, tokenizer.pad_token_id, k, eps)
        self.device = to_device_once(model, device)                       # the buffers U, S, V move with the model
        self.dtype = next(model.parameters()).dtype

    def _ortho(self):
        terms = torch.stack([layer.ortho() for layer in self.layers])
        return terms.sum() if self.ortho_reduce == "sum" else terms.mean()

    def learn(self, inputs, targets, epochs=3, batch=16, lr=1e-3, seed=0, log=None):
        """Learn one new task: train only R_t (all layers), loss = language-model loss + lam * Eq. 8 (paper Eq. 10); then fit the
        router on the task's training inputs. Returns 0 (no input is ever withheld from a task); self.info has the losses."""
        rows = self.ids(inputs)
        for layer in self.layers:
            layer.new_task()
        T = len(self.layers[0].R)
        answers = self._targets(targets)

        def set_route(part):                                              # while training, only the new adapter is on
            p = torch.zeros(len(part), T, dtype=self.dtype)
            p[:, T - 1] = 1.0
            self.router.p = p.to(self.device)                             # CPU -> device

        extra = (lambda: self.lam * self._ortho()) if T > 1 and self.lam > 0 else None
        self.info = dict(zip(("loss", "extra"), self._fit(rows, answers, set_route, epochs, batch, lr, seed, log, extra)))
        self.router.p = None
        self.gmm.add_task(rows, seed)                                     # after the adapter is trained, on the CPU
        return 0

    @torch.no_grad()
    def answer(self, inputs, rule="soft", route=None, batch=16):
        """Returns the answers, the most probable task of every input (1-based) and zeros (nothing is certified).
        rule: 'soft' (the paper: adapters blended with the posterior, Eq. 15), 'hard' (only the most probable task's
        adapter), 'sum' (all adapters added, the paper's no-router ablation: R(x) = sum_t R_t)."""
        assert rule in ("soft", "hard", "sum")
        rows = self.ids(inputs)
        post = self.gmm.posterior(rows)                                   # CPU, float64
        guess = post.argmax(1)
        if route is not None:                                             # a given route (1-based) is used as a hard choice
            guess = torch.as_tensor(route).long() - 1
            weights = torch.nn.functional.one_hot(guess, post.shape[1]).double()
        elif rule == "soft":
            weights = post
        elif rule == "hard":
            weights = torch.nn.functional.one_hot(guess, post.shape[1]).double()
        else:
            weights = torch.ones_like(post)
        self.model.eval()
        out = []
        for s in range(0, len(rows), batch):
            ids, att = self._pad(rows[s:s + batch])                       # on self.device
            self.router.p = weights[s:s + batch].to(device=self.device, dtype=self.dtype)   # CPU float64 -> device
            gen = self.model.generate(input_ids=ids, attention_mask=att, max_new_tokens=self.max_new, do_sample=False,
                                      num_beams=1, pad_token_id=self.tok.pad_token_id)
            out += [t.strip() for t in self.tok.batch_decode(gen, skip_special_tokens=True)]
        self.router.p = None
        return out, guess + 1, torch.zeros(len(rows), dtype=torch.bool)
'''

In [ ]:
FILES["run_five.py"] = r'''"""The five-task comparison: the project's certified switch ("ours") and Latent-LoRA ("latent_lora"), same data, same
training loop, same decoding, same scoring.

    python run_five.py --data /path/to/CL_Benchmark --out /path/to/results          (real run: t5-base, 3 seeds, 2 methods)
    LL_TEST=1 python run_five.py --data ... --out ...                                (tiny run for a CPU: t5-small)
    python run_five.py --summary --out /path/to/results                              (only print the table of finished runs)

Order of the tasks and the five tasks follow certified_switch/demo.py. After every task, the test inputs of every task
learned so far are answered with the original and with three reworded definitions; every answer is saved.
  ours         ContinualModel with rule "certified"; from the same trained adapters also "nearest" and "all" (plain adapters)
  latent_lora  rule "soft" (the paper); from the same trained adapters also "sum" (all adapters added, the paper's ablation)
A self-test (8 training examples, 2 optimiser steps per method, learn, answer, save) runs first, on the real device.
"""
import argparse
import json
import os
import random
import sys
import time
import traceback

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)                                         # latent_lora.py and certified_switch.py next to this file
import torch

TINY = os.environ.get("LL_TEST") == "1"

p = argparse.ArgumentParser()
p.add_argument("--data", default=os.environ.get("LL_DATA", "CL_Benchmark"))
p.add_argument("--out", default=os.environ.get("LL_OUT", os.path.join(HERE, "results_tiny" if TINY else "results")))
p.add_argument("--model", default="t5-small" if TINY else "t5-base")
p.add_argument("--methods", default="ours,latent_lora")
p.add_argument("--seeds", default="42" if TINY else "42,43,44")
p.add_argument("--train", type=int, default=24 if TINY else 800, help="training examples per task (800 = all of train.json)")
p.add_argument("--test", type=int, default=8 if TINY else 100, help="test examples per task")
p.add_argument("--epochs", type=int, default=1 if TINY else 10)
p.add_argument("--batch", type=int, default=8 if TINY else 32, help="examples per optimiser step (effective batch)")
p.add_argument("--ll_batch", type=int, default=8, help="Latent-LoRA examples per optimiser step (the paper's batch size for T5 is 8)")
p.add_argument("--micro", type=int, default=4 if TINY else 16, help="examples per forward pass; batch/micro = gradient accumulation steps")
p.add_argument("--eval_batch", type=int, default=8 if TINY else 32)
p.add_argument("--max_len", type=int, default=512)
p.add_argument("--max_new", type=int, default=50)
p.add_argument("--unrelated_n", type=int, default=30 if TINY else 200, help="inputs per unrelated task (only 'ours' uses them)")
p.add_argument("--ours_lr", type=float, default=3e-4)
p.add_argument("--ours_rank", type=int, default=4)
p.add_argument("--ours_alpha", type=int, default=32)
p.add_argument("--ll_lr", type=float, default=3e-4)
p.add_argument("--ll_rank", type=int, default=32)
p.add_argument("--ll_alpha", type=int, default=16)
p.add_argument("--ll_lambda", type=float, default=0.05)
p.add_argument("--ll_k", type=int, default=5)
p.add_argument("--ll_eps", type=float, default=0.01)
p.add_argument("--ll_ortho_reduce", default="sum", choices=["sum", "mean"])
p.add_argument("--log_every", type=int, default=25)
p.add_argument("--device", default=os.environ.get("LL_DEVICE", ""))
p.add_argument("--summary", action="store_true", help="only print the table of the runs found in --out")
args = p.parse_args()

TASKS = ["task1510_evalution_relation_extraction", "task363_sst2_polarity_classification",
         "task748_glucose_reverse_cause_event_detection", "task1687_sentiment140_classification",
         "task875_emotion_classification"]
UNRELATED = "mnli,cb,qqp,rte,dbpedia,agnews,yahoo,multirc,boolq,wic".split(",")
TEMPLATE = "Definition: {d}\n\nNow complete the following example -\nInput: {x}\nOutput: "
METHODS = {"ours": ["certified", "nearest", "all"], "latent_lora": ["soft", "sum"]}      # first rule = the method's own
REWORDED = json.load(open(next(f for f in (os.path.join(HERE, "rewordings.json"),
                                           os.path.join(HERE, "..", "certified_switch", "rewordings.json")) if os.path.exists(f))))
LEVELS = ["original"] + REWORDED["levels"]


# ---- copied unchanged from certified_switch/demo.py: data reading, prompt, score --------------------------------------
def read(folder, name, split):
    data = json.load(open(os.path.join(args.data, folder, name, split + ".json")))
    d = data["Definition"][0] if isinstance(data["Definition"], list) else data["Definition"]
    return d.strip(), [(i["input"], i["output"] if isinstance(i["output"], list) else [i["output"]]) for i in data["Instances"]]


def prompt(definition, x):
    return TEMPLATE.replace("{d}", definition).replace("{x}", x)


def rouge_l(pred, refs):
    """Rouge-L F-measure on lower-case words, best over the reference answers."""
    def lcs(a, b):
        row = [0] * (len(b) + 1)
        for x in a:
            prev = 0
            for j, y in enumerate(b):
                prev, row[j + 1] = row[j + 1], prev + 1 if x == y else max(row[j + 1], row[j])
        return row[-1]
    best, a = 0.0, pred.lower().split()
    for ref in refs:
        b = ref.lower().split()
        n = lcs(a, b)
        if n:
            pr, rc = n / len(a), n / len(b)
            best = max(best, 2 * pr * rc / (pr + rc))
    return 100.0 * best
# -----------------------------------------------------------------------------------------------------------------------


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def write_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    json.dump(obj, open(tmp, "w"))
    os.replace(tmp, path)                                         # a crash never leaves half a file


# ---------------------------------------------------------------------------------------------- summary of finished runs
def metrics_of(res):
    """Learned, AP, FT, answers changed, right task, per wording and rule, from the saved answers of one run (all tasks done so far)."""
    steps, T, refs = res["steps"], len(res["steps"]), res["references"]
    out = {}
    for level in res["levels"]:
        for rule in res["rules"]:
            s = {}                                                # s[i, j]: score of task j after task i was learned
            for i in range(T):
                for j in range(i + 1):
                    ans = steps[i]["answers"][f"{level}|{rule}|{j + 1}"]["answers"]
                    s[i, j] = sum(rouge_l(a, r) for a, r in zip(ans, refs[j])) / len(refs[j])
            changed = compared = 0
            for j in range(T - 1):
                first = steps[j]["answers"][f"{level}|{rule}|{j + 1}"]["answers"]
                last = steps[T - 1]["answers"][f"{level}|{rule}|{j + 1}"]["answers"]
                changed += sum(a != b for a, b in zip(first, last))
                compared += len(first)
            right = n = 0
            for j in range(T):
                route = steps[T - 1]["answers"][f"{level}|{rule}|{j + 1}"]["route"]
                right += sum(r == j + 1 for r in route)
                n += len(route)
            out[f"{level}|{rule}"] = {
                "tasks_done": T, "learned": sum(s[j, j] for j in range(T)) / T, "AP": sum(s[T - 1, j] for j in range(T)) / T,
                "FT": sum(max(s[i, j] for i in range(j, T - 1)) - s[T - 1, j] for j in range(T - 1)) / (T - 1) if T > 1 else 0.0,
                "FT_vs_learned": sum(s[j, j] - s[T - 1, j] for j in range(T - 1)) / (T - 1) if T > 1 else 0.0,
                "changed": changed, "compared": compared, "right_task": right / n,
                "score_matrix": [[s.get((i, j)) for j in range(T)] for i in range(T)]}
    return out


def summarize(out_dir):
    runs = []
    folder = os.path.join(out_dir, "results")
    for f in sorted(os.listdir(folder)) if os.path.isdir(folder) else []:
        if f.endswith(".json"):
            res = json.load(open(os.path.join(folder, f)))
            if res.get("steps"):
                res["metrics"] = metrics_of(res)
                runs.append(res)
    if not runs:
        print("no finished task found in", folder)
        return None
    summary = {}
    names = {"certified": "certified (ours)", "nearest": "nearest zone", "all": "all adapters on", "soft": "soft blend (paper)", "sum": "sum of adapters"}
    print(f"\n{'method':<13}{'rule':<19}{'wording':<10}{'seeds':>6}{'tasks':>6}{'Learned':>9}{'AP':>8}{'FT':>8}{'answers changed':>18}{'right task':>12}")
    for method, rules in METHODS.items():
        mine = [r for r in runs if r["method"] == method]
        if not mine:
            continue
        for rule in rules:
            for level in LEVELS:
                key = f"{level}|{rule}"
                vals = [r["metrics"][key] for r in mine]
                mean = lambda k: sum(v[k] for v in vals) / len(vals)
                row = {"seeds": [r["seed"] for r in mine], "tasks_done": [v["tasks_done"] for v in vals],
                       "per_seed": {str(r["seed"]): {k: v[k] for k in ("learned", "AP", "FT", "FT_vs_learned", "changed", "compared", "right_task")}
                                    for r, v in zip(mine, vals)},
                       "learned": mean("learned"), "AP": mean("AP"), "FT": mean("FT"), "FT_vs_learned": mean("FT_vs_learned"),
                       "changed": sum(v["changed"] for v in vals), "compared": sum(v["compared"] for v in vals), "right_task": mean("right_task")}
                summary[f"{method}|{rule}|{level}"] = row
                print(f"{method:<13}{names[rule]:<19}{level:<10}{len(vals):>6}{min(row['tasks_done']):>6}{row['learned']:>9.2f}{row['AP']:>8.2f}"
                      f"{row['FT']:>8.2f}{str(row['changed']) + ' of ' + str(row['compared']):>18}{row['right_task']:>12.3f}")
    print("\nLearned: mean score of each task right after it was learned. AP: mean score after the last task. FT: mean over tasks 1..T-1 of"
          "\n(best score after learning it, before the last task) minus (score at the end). Answers changed: test inputs of tasks 1..T-1 whose"
          "\nanswer at the end differs from the answer right after their task was learned (summed over seeds). Right task: share of test"
          "\ninputs at the end that were given to their own task (Latent-LoRA: most probable task). Scores: Rouge-L, mean over seeds.")
    write_json({"settings": runs[0]["settings"], "table": summary}, os.path.join(out_dir, "summary.json"))
    print("summary written to", os.path.join(out_dir, "summary.json"))
    return summary


# ---------------------------------------------------------------------------------------------- one method
def load_base(model_name):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)         # on the CPU; the wrappers move it to the device once
    return model, tok


def unrelated_texts(n):
    return [[prompt(d, x) for x, _ in rows[:n]] for d, rows in (read("Long_Sequence", name, "train") for name in UNRELATED)]


def build(method, model, tok, device, unrelated):
    from latent_lora import CertifiedOnDevice, LatentLoRAModel
    if method == "ours":
        return CertifiedOnDevice(model, tok, unrelated, device=device, micro=args.micro, rank=args.ours_rank, alpha=args.ours_alpha,
                                 max_len=args.max_len, max_new=args.max_new, rule="certified")
    return LatentLoRAModel(model, tok, rank=args.ll_rank, alpha=args.ll_alpha, lam=args.ll_lambda, k=args.ll_k, eps=args.ll_eps,
                           max_len=args.max_len, max_new=args.max_new, device=device, micro=args.micro, ortho_reduce=args.ll_ortho_reduce)


def lr_of(method):
    return args.ours_lr if method == "ours" else args.ll_lr


def batch_of(method):
    return args.batch if method == "ours" else args.ll_batch


def check_device(model, device):
    """Every parameter and buffer of the model must be on the device."""
    wrong = {str(t.device) for t in list(model.parameters()) + list(model.buffers()) if t.device.type != torch.device(device).type}
    assert not wrong, f"tensors of the model on {wrong}, expected {device}"


def selftest(methods, device):
    """8 training examples, one optimiser step per task, two tasks, per method: learn, answer (every rule), save. On the real device."""
    log(f"self-test on {device} ({args.model}) ...")
    t0 = time.time()
    tasks = [(name, read("SuperNI", name, "train"), read("SuperNI", name, "test")) for name in TASKS[:2]]
    unrelated = unrelated_texts(min(args.unrelated_n, 20))
    for method in methods:
        try:
            model, tok = load_base(args.model)
            obj = build(method, model, tok, device, unrelated)
            check_device(model, device)
            saved = {}
            for j, (name, (d, train), (_, test)) in enumerate(tasks):
                train, test = train[:8], test[:8]
                obj.micro = min(obj.micro, 4)                           # also exercises gradient accumulation (8 = 2 pieces of 4)
                obj.learn([prompt(d, x) for x, _ in train], [y[0] for _, y in train], epochs=1, batch=8, lr=lr_of(method), seed=0)
                check_device(model, device)
                for rule in METHODS[method]:
                    ans, route, cert = obj.answer([prompt(d, x) for x, _ in test], rule=rule, batch=4)
                    assert len(ans) == 8 and all(isinstance(a, str) for a in ans) and len(route) == 8, "bad answers"
                    saved[f"{j + 1}|{rule}"] = {"answers": ans, "route": route.tolist()}
            trained = [q for q in model.parameters() if q.requires_grad]
            assert trained and all(float(q.detach().abs().sum()) > 0 for q in trained[-8:]), "the newest adapter did not change in training"
            path = os.path.join(args.out, "selftest", f"{method}.json")
            write_json(saved, path)
            assert json.load(open(path)) == saved, "saved file differs from what was written"
            del obj, model
            log(f"  {method}: ok ({time.time() - t0:.0f} s so far)")
        except Exception:
            print("\n" + "=" * 100 + f"\nSELF-TEST FAILED for method '{method}' on device '{device}'. Nothing of the long run was started.\n"
                  "The error (a device mismatch, an out-of-memory error or a missing file shows here):\n" + "=" * 100, flush=True)
            traceback.print_exc()
            sys.exit(2)
    log(f"self-test passed in {time.time() - t0:.0f} s")


def run_job(method, seed, device):
    path = os.path.join(args.out, "results", f"{method}_seed{seed}.json")
    if os.path.exists(path):
        old = json.load(open(path))
        if old.get("complete"):
            log(f"{method} seed {seed}: already finished, skipped")
            return
    random.seed(seed)
    torch.manual_seed(seed)
    model, tok = load_base(args.model)
    unrelated = unrelated_texts(args.unrelated_n) if method == "ours" else []
    obj = build(method, model, tok, device, unrelated)
    check_device(model, device)
    n_params = sum(q.numel() for q in model.parameters())
    log(f"{method} seed {seed}: {args.model} ({n_params / 1e6:.0f}M frozen numbers), {len(obj.layers)} adapted layers, device {device}")
    res = {"method": method, "seed": seed, "settings": dict(vars(args), device=str(device)), "tasks": TASKS, "levels": LEVELS,
           "rules": METHODS[method], "references": [], "steps": [], "complete": False}
    tests, t0 = [], time.time()
    for j, name in enumerate(TASKS):
        d, train = read("SuperNI", name, "train")
        _, test = read("SuperNI", name, "test")
        train, test = train[:args.train], test[:args.test]
        defs = [d] + REWORDED["tasks"][name.split("_")[0]]
        tests.append({level: [prompt(defs[li], x) for x, _ in test] for li, level in enumerate(LEVELS)})
        res["references"].append([y for _, y in test])
        t1 = time.time()
        blocked = obj.learn([prompt(d, x) for x, _ in train], [y[0] for _, y in train], epochs=args.epochs, batch=batch_of(method),
                            lr=lr_of(method), seed=seed, log=args.log_every)
        t_train = time.time() - t1
        answers = {}
        for k in range(j + 1):                                          # every task learned so far, every wording, every rule
            for level in LEVELS:
                for rule in METHODS[method]:
                    ans, route, cert = obj.answer(tests[k][level], rule=rule, batch=args.eval_batch)
                    answers[f"{level}|{rule}|{k + 1}"] = {"answers": ans, "route": route.tolist(), "certified": cert.tolist()}
        step = {"task_index": j + 1, "task": name, "blocked_training_inputs": blocked, "train_minutes": t_train / 60,
                "minutes_total": (time.time() - t0) / 60, "loss_last_epoch": obj.info["loss"], "extra_term_last_epoch": obj.info["extra"],
                "micro_batch": obj.micro, "answers": answers}
        if method == "ours":
            step["zone"] = {"rho": obj.zones.tasks[-1]["rho"], "opening_tokens": obj.zones.tasks[-1]["n"]}
        res["steps"].append(step)
        res["complete"] = j == len(TASKS) - 1
        write_json(res, path)                                           # after every task
        m = metrics_of(res)[f"original|{METHODS[method][0]}"]
        log(f"{method} seed {seed}: task {j + 1}/{len(TASKS)} ({name.split('_')[0]}) learned; train {t_train / 60:.1f} min, answers done, "
            f"{(time.time() - t0) / 60:.1f} min so far; loss {obj.info['loss']:.3f}"
            + (f", orthogonal term {obj.info['extra']:.4g}" if obj.info["extra"] is not None else "")
            + f"; original wording, {METHODS[method][0]}: learned {m['learned']:.1f}, now {m['AP']:.1f}, changed {m['changed']} of {m['compared']}")
    res["metrics"] = metrics_of(res)
    write_json(res, path)
    log(f"{method} seed {seed}: finished in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    if args.summary:
        summarize(args.out)
        sys.exit(0)
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "0")) or torch.get_num_threads())
    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    if device.startswith("cuda") and not torch.cuda.is_available():
        sys.exit("CUDA was asked for but PyTorch sees no GPU.")
    methods = [m for m in args.methods.split(",") if m]
    seeds = [int(s) for s in args.seeds.split(",") if s]
    assert all(m in METHODS for m in methods), f"methods must be from {list(METHODS)}"
    os.makedirs(args.out, exist_ok=True)
    log(f"device {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.startswith("cuda") else "")
        + f"; methods {methods}; seeds {seeds}; {'TINY MODE' if TINY else 'full settings'}: {args.model}, {args.train} train / {args.test} test per task, "
          f"{args.epochs} epochs, batch {args.batch} for ours and {args.ll_batch} for latent_lora (pieces of {args.micro})")
    selftest(methods, device)
    for seed in seeds:
        for method in methods:
            try:
                run_job(method, seed, device)
            except Exception:
                traceback.print_exc()
                log(f"{method} seed {seed}: FAILED (see above); the tasks finished before the error are in the result file")
                sys.exit(1)
    summarize(args.out)
'''

In [ ]:
FILES["rewordings.json"] = r'''{
 "note": "Reworded task definitions (light, medium, heavy), written by machine for the tests. Answer formats are unchanged.",
 "levels": [
  "light",
  "medium",
  "heavy"
 ],
 "tasks": {
  "task1510": [
   "Given a phrase that describes the relationship between two words, extract the two words and the lexical relation between them. The relation must be one of the types 'MemberOf', 'MadeOf', 'Synonym', 'Entails', 'HasA', 'HasProperty', 'PartOf', 'Antonym' or 'IsA'. The output should be in the format: word1 relation word2.",
   "You will see a short phrase that tells how two words are related. Find the two words and name the lexical relation that links them. Allowed relations are 'MemberOf', 'MadeOf', 'Synonym', 'Entails', 'HasA', 'HasProperty', 'PartOf', 'Antonym' and 'IsA'. Write your answer as: word1 relation word2.",
   "Read the phrase, pick out the pair of words it talks about, and say which link holds between them ('MemberOf', 'MadeOf', 'Synonym', 'Entails', 'HasA', 'HasProperty', 'PartOf', 'Antonym', 'IsA'). Answer format: word1 relation word2."
  ],
  "task363": [
   "In this task, you are given a sentence from a movie review. Your task is to classify the sentence as \"POS\" if its sentiment is positive or as \"NEG\" if its sentiment is negative.",
   "You will read a sentence taken from a film review. Decide whether it expresses a positive or a negative opinion. Answer \"POS\" for positive and \"NEG\" for negative.",
   "Here is one line from a review of a movie. Label it \"POS\" when the writer liked what they saw and \"NEG\" when they did not."
  ],
  "task748": [
   "In this task, you are given a short story, and one sentence from the story is selected. Consider the events that happen after that sentence. Is any of them directly caused by it or made possible by it? Write your answer in the form \" A >causes/enables> B\". Try to use phrases and sentences from the story in your answer when possible. Do not change the selected sentence in your answer.",
   "You will read a short story in which one sentence has been picked out. Think about what happens later in the story and find an event that the picked sentence directly causes or makes possible. Give the answer as \" A >causes/enables> B\", reusing wording from the story where you can and keeping the picked sentence unchanged.",
   "A brief story follows, with one of its sentences marked. Name a later event that happens because of the marked sentence, or that it allows to happen. Use the format \" A >causes/enables> B\", copy text from the story when possible, and leave the marked sentence as it is."
  ],
  "task1687": [
   "In this task, you are given the text of a tweet. Your task is to classify the given tweet text into one of two categories, 1) positive or 2) negative, based on its content.",
   "You will read the text of a tweet. Decide from what it says whether the tweet is positive or negative, and answer with one of these two categories.",
   "Below is a post from Twitter. Say whether its mood is positive or negative."
  ],
  "task875": [
   "In this task, you are given a sentence that contains a particular emotion. You must classify the sentence into one of these emotions: 'joy', 'love', 'anger', 'fear', or 'surprise'.",
   "You will read a sentence that expresses a feeling. Choose which emotion it shows from this list: 'joy', 'love', 'anger', 'fear', 'surprise'.",
   "Which feeling does the writer of this sentence have? Pick one of 'joy', 'love', 'anger', 'fear', 'surprise'."
  ]
 }
}'''

In [ ]:
for name, text in FILES.items():
    with open(os.path.join(CODE, name), "w") as f:
        f.write(text)
print("written to", CODE, ":", ", ".join(f"{n} ({len(t.splitlines())} lines)" for n, t in FILES.items()))

## Machine, data, model

In [ ]:
NO_INTERNET = ("NO INTERNET: the notebook cannot download what it needs ({what}). On Kaggle the internet is off until you switch it on: panel on "
               "the right -> Session options (or Notebook options) -> Internet -> on. The switch appears only after your phone number is "
               "verified (kaggle.com -> Settings -> Phone verification). Then run all again.")

gpu_lines = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip().splitlines()
if not NB_TEST:
    assert gpu_lines, ("No GPU. On Kaggle: panel on the right -> Session options -> Accelerator -> GPU T4 x2 (or P100). Then run all again.")
N_GPUS = len(gpu_lines)
WORKERS = int(os.environ.get("LL_NB_WORKERS") or (N_GPUS if N_GPUS else 1))    # one job per GPU
print(f"GPUs seen: {N_GPUS}" + ("" if not gpu_lines else " -> " + " | ".join(gpu_lines)))
print(f"{WORKERS} job(s) at a time" + (", one per GPU" if not NB_TEST else " (test mode, CPU)"))

DATA = os.environ.get("LL_NB_DATA")
if not DATA:
    repo = os.path.join(WORK, "gainlora")
    if not os.path.isdir(os.path.join(repo, "CL_Benchmark", "SuperNI")):
        shutil.rmtree(repo, ignore_errors=True)
        r = subprocess.run(["git", "clone", CFG["repo_url"], repo], capture_output=True, text=True)
        if r.returncode != 0:
            import socket
            try:
                socket.create_connection(("github.com", 443), timeout=10).close()
            except OSError:
                raise RuntimeError(NO_INTERNET.format(what="git clone " + CFG["repo_url"])) from None
            raise RuntimeError("git clone failed although github.com can be reached:\n" + (r.stdout + r.stderr)[-2000:])
        subprocess.run(["git", "-C", repo, "checkout", CFG["repo_commit"]], check=True, capture_output=True, text=True)
    DATA = os.path.join(repo, "CL_Benchmark")
TASKS = ["task1510_evalution_relation_extraction", "task363_sst2_polarity_classification", "task748_glucose_reverse_cause_event_detection",
         "task1687_sentiment140_classification", "task875_emotion_classification"]
for t in TASKS:
    for split in ("train", "test"):
        assert os.path.exists(os.path.join(DATA, "SuperNI", t, split + ".json")), f"missing data file {t}/{split}.json in {DATA}"
for t in "mnli,cb,qqp,rte,dbpedia,agnews,yahoo,multirc,boolq,wic".split(","):
    assert os.path.exists(os.path.join(DATA, "Long_Sequence", t, "train.json")), f"missing Long_Sequence/{t}/train.json in {DATA}"
print("data:", DATA)

# the model and tokenizer, downloaded once here (so that the parallel jobs do not download at the same time)
check = ("import torch, transformers; from transformers import AutoModelForSeq2SeqLM, AutoTokenizer; "
         f"AutoTokenizer.from_pretrained('{CFG['model']}'); AutoModelForSeq2SeqLM.from_pretrained('{CFG['model']}'); "
         "print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())")
r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
if r.returncode != 0 and "sentencepiece" in (r.stdout + r.stderr).lower():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], capture_output=True, text=True)
    r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
if r.returncode != 0:
    text = r.stdout + r.stderr
    if any(w in text for w in ("ConnectionError", "Max retries", "Temporary failure", "Name or service", "offline", "LocalEntryNotFound", "couldn't connect", "Can't load")):
        raise RuntimeError(NO_INTERNET.format(what=f"the {CFG['model']} weights from huggingface.co") + "\n\nDetails:\n" + text[-1500:])
    raise RuntimeError("loading the model failed:\n" + text[-3000:])
print(r.stdout.strip())

## Time estimate
Rough guess, not measured on a GPU: about 45 minutes for one (method, seed) job on a T4 (10 epochs x 800 examples x 5 tasks of training, plus
re-answering about 6000 test inputs per rule after the tasks; the earlier notebook needed 36 minutes per method for a similar setting).

In [ ]:
jobs = [(m, s) for s in CFG["seeds"] for m in CFG["methods"]]
if not NB_TEST:
    rounds = -(-len(jobs) // WORKERS)
    print(f"{len(jobs)} jobs ({len(CFG['methods'])} methods x {len(CFG['seeds'])} seeds), {WORKERS} at a time, about {CFG['minutes_per_job']} min each: "
          f"estimated total about {rounds * CFG['minutes_per_job'] / 60:.1f} hours (estimate, not measured).")
else:
    print(f"{len(jobs)} jobs in tiny test mode")

## The runs
One process per job, one log file per job in `logs/`. A progress line per running job is printed every few minutes.

In [ ]:
FAILED, RUNNING, ABORT = {}, {}, []
lock = threading.Lock()
queue_ = list(jobs)


def tail(path, n=1):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - 6000))
            lines = [l for l in f.read().decode(errors="replace").splitlines() if l.strip()]
        return lines[-n:]
    except OSError:
        return []


def run_job(job, gpu):
    method, seed = job
    log_path = os.path.join(LOGS, f"{method}_seed{seed}.log")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
    if NB_TEST:
        env["LL_TEST"] = "1"
    cmd = [sys.executable, os.path.join(CODE, "run_five.py"), "--data", DATA, "--out", OUT, "--methods", method, "--seeds", str(seed)]
    t0 = time.time()
    with open(log_path, "a") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=CODE)
        with lock:
            RUNNING[job] = (proc, log_path, t0, gpu)
        code = proc.wait()
    with lock:
        RUNNING.pop(job, None)
        if code != 0:
            FAILED[job] = code
            if code == 2:                                            # the self-test failed: the same would happen to every job
                ABORT.append(job)
    print(f"[GPU {gpu}] {method} seed {seed}: {'finished' if code == 0 else 'FAILED (exit code %d)' % code} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if code != 0:
        print("   the end of its log:\n   " + "\n   ".join(tail(log_path, 40)), flush=True)


def worker(gpu):
    while True:
        with lock:
            if not queue_ or ABORT:
                return
            job = queue_.pop(0)
        run_job(job, gpu)


def reporter(stop):
    while not stop.wait(CFG["progress_every_s"]):
        with lock:
            now = list(RUNNING.items())
        for (m, s), (proc, path, t0, gpu) in now:
            last = tail(path)
            print(f"  ... GPU {gpu}, {m} seed {s}, {(time.time() - t0) / 60:.0f} min: {last[0][:200] if last else '(no output yet)'}", flush=True)


print(f"starting {len(jobs)} jobs on {WORKERS} worker(s); the first thing every job does is a self-test on its GPU", flush=True)
stop = threading.Event()
threading.Thread(target=reporter, args=(stop,), daemon=True).start()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()
stop.set()
if ABORT:
    print("\n" + "=" * 90 + "\nTHE SELF-TEST FAILED, so no further job was started. Its message is printed above (the end of the log). "
          "Copy it into the chat.\n" + "=" * 90)

## Results
**Learned**: mean score of each task right after it was learned. **AP**: mean score after the last task. **FT**: forgetting.
**Answers changed**: test inputs of tasks 1 to 4 whose answer at the end differs from the answer right after their task was learned
(for "ours" with rule "certified" this is 0 by construction). Scores are Rouge-L, mean over the seeds.

In [ ]:
r = subprocess.run([sys.executable, os.path.join(CODE, "run_five.py"), "--summary", "--out", OUT], capture_output=True, text=True, cwd=CODE,
                   env=dict(os.environ, LL_TEST="1") if NB_TEST else None)
print(r.stdout + r.stderr[-1500:])
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:       # the small result files only; there are no model weights anywhere
    for root, _, files in os.walk(OUT):
        for f in files:
            if f.endswith((".json", ".log")):
                z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), os.path.dirname(OUT.rstrip("/"))))
print("zip with all .json and .log files:", ZIP, f"({os.path.getsize(ZIP) / 1e6:.1f} MB)")
if FAILED:
    print("\nJOBS THAT FAILED (their finished tasks are in results/; run all again to repeat the unfinished jobs):")
    for (m, s), c in FAILED.items():
        print(f"   {m} seed {s}: exit code {c}, log {os.path.join(LOGS, f'{m}_seed{s}.log')}")
else:
    print("\nall jobs finished")